In [2]:
# the data in the csv was sorted by 'niet omstreden' (non-contentious) asc first and then by 'omstreden' (contentious) desc

import pandas as pd

with open('./gpt and llama/gpt-4o  ori_en_C.csv', 'r') as f:
     vote_data = pd.read_csv(f)

with open("./gpt-4o_temperature1_ori-en-CandR.csv", 'r') as f:
     vote_data_gpt = pd.read_csv(f)

with open("./llama3-70b_temperature1_ori-en-CandR.csv", 'r') as f:
     vote_data_llama = pd.read_csv(f)

# Convert the 'score' column to proportion of each score for each target as reference, and then sort by 1.0 (contentious) desc
def make_proportion_reference(data):
    # convert '0.5' in the 'score' column to 1
    data['score'] = data['score'].apply(lambda x: 1 if x == 0.5 else x)

    # group the data by 'target' and get the number of lines for each group
    grouped = data.groupby('target')['score'].value_counts(normalize=True).unstack().fillna(0)
    # , then subgroup by 'score' and get the number and proportion of lines for each subgroup.
    grouped['total'] = data.groupby('target')['score'].count()
    proportion_data = grouped.reset_index()

    # sort by 1.0 (contentious) desc
    proportion_data = proportion_data.sort_values(by=[1.0], ascending=True)

    # get list of target
    target_list = proportion_data['target'].tolist()

    return proportion_data, target_list

# Convert the 'output_mv' column to proportion of each score for each target as prediction, and then sort by the same target value order as reference
def make_proportion_prediction(data, target_list):
    # convert '0.5' in the 'score' column to 1
    data['output_mv'] = data['output_mv'].apply(lambda x: 1 if x == 0.5 else x)

    # group the data by 'target' and get the number of lines for each group
    grouped = data.groupby('target')['output_mv'].value_counts(normalize=True).unstack().fillna(0)
    # , then subgroup by 'score' and get the number and proportion of lines for each subgroup.
    grouped['total'] = data.groupby('target')['output_mv'].count()
    proportion_data = grouped.reset_index()

    # sort to keep the same target value order as target_list
    proportion_data = proportion_data.sort_values(by=['target'], key=lambda x: x.map({v: i for i, v in enumerate(target_list) }))
    return proportion_data

# get the proportion data for reference and prediction
proportion_data, target_list = make_proportion_reference(vote_data)
proportion_data_gpt = make_proportion_prediction(vote_data_gpt, target_list)
proportion_data_llama = make_proportion_prediction(vote_data_llama, target_list)

# add all numbers in total column
total = proportion_data['total'].sum()
total

print(total)

proportion_data

2090


score,target,0.0,1.0,total
65,ontdekken,1.0,0.0,20
26,historisch,1.0,0.0,118
82,westers,1.0,0.0,8
76,stam,1.0,0.0,17
31,indisch,1.0,0.0,20
...,...,...,...,...
29,inboorling,0.0,1.0,16
16,bosneger,0.0,1.0,9
13,birma,0.0,1.0,1
63,negerslaven,0.0,1.0,22


In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

# prepare data for plotting
df_human = proportion_data.copy()
df_human["group"] = "human annotators"
df_gpt = proportion_data_gpt.copy()  
df_gpt["group"] = 'gpt-4o'
df_llama = proportion_data_llama.copy()
df_llama["group"] = 'llama3-70b'

df_human["target_labeled"] = df_human["target"] + " (n=" + df_human["total"].astype(str) + ")"
df_gpt["target_labeled"] = df_gpt["target"] + " (n=" + df_gpt["total"].astype(str) + ")"
df_llama["target_labeled"] = df_llama["target"] + " (n=" + df_llama["total"].astype(str) + ")"

df_human = df_human[["target", "target_labeled", 1.0, "group"]]
df_gpt = df_gpt[["target", "target_labeled", 1.0, "group"]]
df_llama = df_llama[["target", "target_labeled", 1.0, "group"]]

# Add bars to the subplot
def add_bars(fig, df, targets, name, color, row, col, showlegend=True):
    sub_df = df[df["target_labeled"].isin(targets)]
    fig.add_trace(
        go.Bar(
            x=sub_df[1.0],
            y=sub_df["target_labeled"],
            name=name,
            orientation="h",
            marker_color=color,
            text=sub_df[1.0],
            textposition="outside",
            marker_line_width=0.5,
            marker_line_color="black",
            showlegend=showlegend
        ),
        row=row, col=col
    )

# Get the target order from human annotators and split into 4 columns
target_order = df_human["target_labeled"].tolist()
q = len(target_order) // 4
targets_col4 = target_order[:q]
targets_col3 = target_order[q:2*q]
targets_col2 = target_order[2*q:3*q]
targets_col1 = target_order[3*q:]

# Create the combined plot with 4 columns
def make_combined_plot(t1, t2, t3, t4):
    max_len = max(len(t1), len(t2), len(t3), len(t4))

    fig = make_subplots(
        rows=1, cols=4,
        shared_xaxes=False,
        horizontal_spacing=0.22,
        subplot_titles=None
    )

    # first column
    add_bars(fig, df_llama, t1, "llama3-70b",       "lightgrey", 1, 1, showlegend=True)
    add_bars(fig, df_gpt,   t1, "gpt-4o",           "grey",      1, 1, showlegend=True)
    add_bars(fig, df_human, t1, "human annotators", "black",     1, 1, showlegend=True)

    # second column
    add_bars(fig, df_llama, t2, "llama3-70b",       "lightgrey", 1, 2, showlegend=False)
    add_bars(fig, df_gpt,   t2, "gpt-4o",           "grey",      1, 2, showlegend=False)
    add_bars(fig, df_human, t2, "human annotators", "black",     1, 2, showlegend=False)

    # third column
    add_bars(fig, df_llama, t3, "llama3-70b",       "lightgrey", 1, 3, showlegend=False)
    add_bars(fig, df_gpt,   t3, "gpt-4o",           "grey",      1, 3, showlegend=False)
    add_bars(fig, df_human, t3, "human annotators", "black",     1, 3, showlegend=False)

    # fourth column
    add_bars(fig, df_llama, t4, "llama3-70b",       "lightgrey", 1, 4, showlegend=False)
    add_bars(fig, df_gpt,   t4, "gpt-4o",           "grey",      1, 4, showlegend=False)
    add_bars(fig, df_human, t4, "human annotators", "black",     1, 4, showlegend=False)

    fig.update_layout(
        barmode="group",
        bargap=0.2,
        width=2100,
        height=55 * max_len,
        font=dict(size=20),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=60, r=60, t=60, b=40),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.01,
            xanchor="left",
            x=0,
            font=dict(size=20),
            title_text=""
        ),
        legend_traceorder="reversed",
        uniformtext_mode="show"
    )

    # set the same y-axis order for all columns based on the target order from human annotators
    fig.update_yaxes(categoryorder="array", categoryarray=t1, row=1, col=1)
    fig.update_yaxes(categoryorder="array", categoryarray=t2, row=1, col=2)
    fig.update_yaxes(categoryorder="array", categoryarray=t3, row=1, col=3)
    fig.update_yaxes(categoryorder="array", categoryarray=t4, row=1, col=4)

    # set x-axis title for each column
    for c in range(1, 5):
        fig.update_xaxes(title_text="Proportion of Contentious Class", row=1, col=c)

    fig.update_traces(cliponaxis=False, texttemplate='%{text:.2f}')
    pio.write_image(fig, "./per_term_mv_vs_mv_4_column.pdf",
                    width=fig.layout.width, height=fig.layout.height)
    fig.show()

make_combined_plot(targets_col1, targets_col2, targets_col3, targets_col4)

In [6]:
# count for how many targets gpt is more contentious than human annotators, and how many targets gpt is less contentious than human annotators, and how many targets gpt is the same as human annotators.
comparison = df_gpt.merge(df_human, on="target", suffixes=("_gpt", "_human"))
comparison["gpt_more_contentious"] = comparison["1.0_gpt"] > comparison["1.0_human"]
comparison["gpt_less_contentious"] = comparison["1.0_gpt"] < comparison["1.0_human"]
num_gpt_more = comparison["gpt_more_contentious"].sum()
num_gpt_less = comparison["gpt_less_contentious"].sum()
num_gpt_same = len(comparison) - num_gpt_more - num_gpt_less
print(f"GPT is more contentious than human annotators for {num_gpt_more} targets.")
print(f"GPT is less contentious than human annotators for {num_gpt_less} targets.")
print(f"GPT is the same as human annotators for {num_gpt_same} targets.")


GPT is more contentious than human annotators for 44 targets.
GPT is less contentious than human annotators for 11 targets.
GPT is the same as human annotators for 33 targets.


In [7]:
# count for how many targets gpt is more contentious than human annotators, and how many targets gpt is less contentious than human annotators, and how many targets gpt is the same as human annotators.
comparison_llama = df_llama.merge(df_human, on="target", suffixes=("_llama", "_human"))
comparison_llama["llama_more_contentious"] = comparison_llama["1.0_llama"] > comparison_llama["1.0_human"]
comparison_llama["llama_less_contentious"] = comparison_llama["1.0_llama"] < comparison_llama["1.0_human"]
num_llama_more = comparison_llama["llama_more_contentious"].sum()
num_llama_less = comparison_llama["llama_less_contentious"].sum()
num_llama_same = len(comparison_llama) - num_llama_more - num_llama_less
print(f"Llama is more contentious than human annotators for {num_llama_more} targets.")
print(f"Llama is less contentious than human annotators for {num_llama_less} targets.")
print(f"Llama is the same as human annotators for {num_llama_same} targets.")

Llama is more contentious than human annotators for 52 targets.
Llama is less contentious than human annotators for 13 targets.
Llama is the same as human annotators for 23 targets.


In [8]:
# count for how many targets gpt is the same as llama.
comparison_gpt_llama = df_gpt.merge(df_llama, on="target", suffixes=("_gpt", "_llama"))
comparison_gpt_llama["gpt_same_llama"] = comparison_gpt_llama["1.0_gpt"] == comparison_gpt_llama["1.0_llama"]
num_gpt_same_llama = comparison_gpt_llama["gpt_same_llama"].sum()
num_gpt_diff_llama = len(comparison_gpt_llama) - num_gpt_same_llama
print(f"GPT is the same as Llama for {num_gpt_same_llama} targets.")
print(f"GPT is different from Llama for {num_gpt_diff_llama} targets.")

GPT is the same as Llama for 39 targets.
GPT is different from Llama for 49 targets.
